# Incremental Capstone 13 - Natural Language Processing

The objective of this capstone is to analyze reviews, extract insights, and understand sentiment.</br>
</br>
IMDB receives thousands of customer reviews and feedback. However, manually analyzing this data is inefficient. </br>
The goal of this capstone is to develop an NLP-powered sentiment analysis system that automatically classifies </br>
reviews as positive or negative.

## Setup

### Imports

In [98]:
import nltk
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

import textwrap
from collections import Counter
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords, movie_reviews
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from wordcloud import WordCloud

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('vader_lexicon', quiet=True)

True

### Configuration

In [99]:
# GloVe download settings
glove_dir = '../data'
glove_url = 'https://nlp.stanford.edu/data/glove.twitter.27B.zip'

# Download NLTK resources
nltk.download('stopwords', quiet=True)

# Set up stop words and tokenizer
stop_words = set(stopwords.words('english'))

# Need a tokenizer here..
# tweet_tokenizer = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=True)

# Force CPU only
tf.config.set_visible_devices([], 'GPU')

### EDA Utility Functions

In [100]:
data_set_name = ''
target_label = ''

def WrapText(text, max_width=15):
    """Wrap text to multiple lines"""
    words = str(text).split()
    lines = []
    current_line = ""
    
    for word in words:
        if len(current_line + " " + word) <= max_width:
            current_line = (current_line + " " + word).strip()
        else:
            if current_line:
                lines.append(current_line)
            current_line = word
    if current_line:
        lines.append(current_line)
    
    return "\n".join(lines)

def DisplayTable(df_target, table_title=None, max_cell_length=30, show_index=False, 
                 wrap_headers=True, header_wrap_width=15, min_height=3,
                 row_height=0.35, font_size=10):

    if df_target.empty:
        print(f"No data to display{': ' + table_title if table_title else ''}")
        return

    n_rows, n_cols = df_target.shape
    
    # Adjust columns if showing index
    if show_index:
        n_cols += 1

    # Calculate width based on longest column name or cell content
    col_widths = []
    for col in df_target.columns:
        if wrap_headers:
            max_len = max(len(line) for line in WrapText(col, header_wrap_width).split('\n'))
        else:
            max_len = len(str(col))
        for val in df_target[col]:
            val_len = len(f"{val:.2f}" if isinstance(val, float) else str(val))
            max_len = max(max_len, val_len)
        col_widths.append(min(max_len, max_cell_length))
    
    # Dynamic figure width based on content
    fig_width = max(sum(col_widths) * 0.15, n_cols * 2.0)
    
    # Calculate extra height for wrapped headers
    if wrap_headers:
        max_header_lines = max(len(WrapText(col, header_wrap_width).split('\n')) for col in df_target.columns)
    else:
        max_header_lines = 1
    
    fig_height = (n_rows + max_header_lines) * row_height
    if table_title:
        fig_height += 0.4
    
    # Ensure minimum height for small tables
    fig_height = max(fig_height, min_height)
    
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis('off')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    # Format floats and truncate long text
    cell_text = []
    for idx, row in zip(df_target.index, df_target.values):
        new_row = []
        
        # Add index as first column if show_index
        if show_index:
            s = str(idx)
            if len(s) > max_cell_length:
                s = s[:max_cell_length - 3] + '...'
            new_row.append(s)
        
        for val in row:
            if isinstance(val, (float)) and not isinstance(val, bool):
                new_row.append(f"{val:.2f}")
            else:
                s = str(val)
                if len(s) > max_cell_length:
                    s = s[:max_cell_length - 3] + '...'
                new_row.append(s)
        cell_text.append(new_row)

    # Build column labels (with wrapping)
    if show_index:
        col_labels = [df_target.index.name or '']
        if wrap_headers:
            col_labels += [WrapText(col, header_wrap_width) for col in df_target.columns]
        else:
            col_labels += list(df_target.columns)
    else:
        if wrap_headers:
            col_labels = [WrapText(col, header_wrap_width) for col in df_target.columns]
        else:
            col_labels = list(df_target.columns)

    table = ax.table(
        cellText=cell_text,
        colLabels=col_labels,
        cellLoc='center',
        loc='upper center',
        bbox=[0, 0, 1, 1]
    )

    # Bold column headers and set header background
    for col in range(n_cols):
        table[(0, col)].set_facecolor('#4a90d9')
        table[(0, col)].set_text_props(color='white', fontweight='bold')

    # Style data cells
    for row in range(1, n_rows + 1):
        for col in range(n_cols):
            if show_index and col == 0:
                table[(row, col)].set_facecolor('#b8d4e8')
                table[(row, col)].set_text_props(fontweight='bold')
            else:
                table[(row, col)].set_facecolor('#d4e6f1')

    table.auto_set_font_size(False)
    table.set_fontsize(font_size)
    table.auto_set_column_width(col=list(range(n_cols)))

    if table_title is not None:
        fig.suptitle(table_title, fontweight='bold', fontsize=14)

    plt.tight_layout(rect=[0, 0, 1, 0.95] if table_title else [0, 0, 1, 1])

    plt.show()
    plt.close(fig)
    print()
    print()
    
def PrintDataFrameStatistics(df_target, display_table=True):
    # Print Stats
    print("***********************************")
    print("Description Stats")
    print("***********************************")
    print()

    # Capture for Table plot
    df_stats = df_target.describe(include='all').T.reset_index()
    df_stats.rename(columns={'index': 'Feature'}, inplace=True)
    print(df_stats)
    print()

    # Print df Column Info
    print("***********************************")
    print("Basic Info of imported data set")
    print("***********************************")
    print()

    df_info = pd.DataFrame({
                            'Feature': df_target.columns,
                            'Non-Null Count': df_target.notna().sum().values,
                            'Null Count': df_target.isna().sum().values,
                            'Dtype': df_target.dtypes.values
                            }).reset_index(drop=True)

    print(f'Dataset Shape:{df_target.shape}')
    df_shape = pd.DataFrame({'Rows': [df_target.shape[0]], 'Columns': [df_target.shape[1]]})
    print()

    print('Do we have any features with null values?:')
    print(df_target.isnull().any().any())
    print()

    print('Do we have any features empty strings?:')
    print((df_target == "").any())
    print()

    print('Feature Columns with that have null values:')
    print(df_target.isnull().sum()[df_target.isnull().sum() > 0])

    # Capture for Table plot
    cols_to_plot = df_target.select_dtypes(exclude=['number']).columns
    missing = df_target[cols_to_plot].isnull().sum()
    missing = missing[missing > 0]
    df_missing = pd.DataFrame({
                                'Feature': missing.index,
                                'Missing Count': missing.values,
                                'Missing %': (missing.values / len(df_target) * 100).round(2)
                                }).reset_index(drop=True)
    print()

    print('Do we have any features with nan values?:')

    cols_to_plot = df_target.select_dtypes(include=['number']).columns
    nan_vals = df_target[cols_to_plot].isna().sum()
    nan_vals = nan_vals[nan_vals > 0]
    # Capture for Table plot
    df_nan = pd.DataFrame({
                            'Feature': nan_vals.index,
                            'NaN Count': nan_vals.values,
                            'NaN %': (nan_vals.values / len(df_target) * 100).round(2)
                        }).reset_index(drop=True)
    print(df_target.isna().any().any())

    # Sum up the number of missing features per row
    missing_per_row = df_target.isna().sum(axis=1)  # count missing per row
    missing_counts = missing_per_row.value_counts().sort_index()  # count rows for each missing count

    df_missingfeature_rowcounts = pd.DataFrame({
        'Missing Features': missing_counts.index,
        'Row Count': missing_counts.values
    })

    print("***********************************")
    print("First 20 rows of Data")
    print("***********************************")
    print()
    print(df_target.head(20))
    print()

    print("***********************************")
    print("First 20 rows of Random Sample Data")
    print("***********************************")
    print()
    df_randomsample = df_target.sample(n=20)
    print(df_target.sample(20))
    print()

    #
    # Display Results in Pretty Tables
    #
    if display_table == True:
        print('Display Analysis Results in Tables')
        DisplayTable(df_stats, f'{data_set_name} Description Statistics')
        print()
        DisplayTable(df_info, f'{data_set_name} Basic Information')
        print()
        DisplayTable(df_shape, f'{data_set_name} Dataset Shape')
        print()
        DisplayTable(df_missing, f'{data_set_name} Missing Categorical (String) Data')
        print()
        DisplayTable(df_nan, f'{data_set_name} Missing Numeric Data')
        print()
        DisplayTable(df_missingfeature_rowcounts, f'{data_set_name} Summary of Missing Feature Row Counts')
        print()    
        DisplayTable(df_randomsample, f'{data_set_name} Random Data Sample')
        
    print()

## **1.0 Examining Sample Data from the Labeled Review Dataset** ##
## Text Preprocessing  ##

### 1.1 Tokenization
Tokenization splits text into individual units (tokens) such as words or sentences.</br>
NLTK [word_tokenize](https://www.nltk.org/api/nltk.tokenize.html) documentation
</br>
For starters, let's tokenize and have a look at a sample (first 500 characters of the first review [0])</br>
This is just a way to get a feel for how these methods work, what they do and what the data generally looks like.

In [ ]:
df_labeledMovieReviews = pd.read_csv('../../../data/Capstone13/labeledTrainData.tsv', sep='\t')
df_unlabeledTestReviews = pd.read_csv('../../../data/Capstone13/unlabeledTestData.tsv', sep='\t')
df_labeledMovieReviews_Original = df_labeledMovieReviews.copy(deep=True)

PrintDataFrameStatistics(df_labeledMovieReviews)
print(df_labeledMovieReviews.columns)

# Sample text for initial investigation
sample_text = df_labeledMovieReviews['review'].iloc[0][:500]
print(type(sample_text))

# Word tokenization
word_tokens = word_tokenize(sample_text)
print(type(word_tokens))

# Sentence tokenization
sent_tokens = sent_tokenize(sample_text)

print(f'Original text:\n{sample_text}\n')
print(f'Word tokens ({len(word_tokens)} tokens):\n{word_tokens[:20]}...\n')
print(f'Sentence tokens ({len(sent_tokens)} sentences):\n{sent_tokens[:2]}')

NameError: name 'df_labeledMoveReviews' is not defined

### 1.2. Normalization and Cleaning

Text normalization includes lowercasing, removing punctuation, and filtering stopwords.

In [ ]:
# Lowercase
tokens_lower = [token.lower() for token in word_tokens]

# Remove non-alphabetic tokens
tokens_alpha = [token for token in tokens_lower if token.isalpha()]

# Remove stopwords
stop_words = set(stopwords.words('english'))
tokens_clean = [token for token in tokens_alpha if token not in stop_words]

print(f'Original tokens: {len(word_tokens)}')
print(f'After lowercasing: {len(tokens_lower)}')
print(f'After removing non-alpha: {len(tokens_alpha)}')
print(f'After removing stopwords: {len(tokens_clean)}')
print(f'\nCleaned tokens:\n{tokens_clean[:15]}')

### 1.3 Stemming and Lemmatization

Stemming reduces words to their root form by removing suffixes. Lemmatization reduces words to their dictionary form (lemma).

NLTK [PorterStemmer](https://www.nltk.org/api/nltk.stem.porter.html) and [WordNetLemmatizer](https://www.nltk.org/api/nltk.stem.wordnet.html) documentation

In [ ]:
# Stemming
stemmer = PorterStemmer()
tokens_stemmed = [stemmer.stem(token) for token in tokens_clean]

# Lemmatization
lemmatizer = WordNetLemmatizer()
tokens_lemmatized = [lemmatizer.lemmatize(token) for token in tokens_clean]

# Compare results
comparison_df = pd.DataFrame({
    'original': tokens_clean[:50],
    'stemmed': tokens_stemmed[:50],
    'lemmatized': tokens_lemmatized[:50]
})

comparison_df

### 1.2 Normalization and cleaning of Sample Data

Text normalization includes lowercasing, removing punctuation, and filtering stopwords.

In [ ]:
# Lowercase
tokens_lower = [token.lower() for token in word_tokens]

# Remove non-alphabetic tokens
tokens_alpha = [token for token in tokens_lower if token.isalpha()]

# Remove stopwords
stop_words = set(stopwords.words('english'))

# To handle embedded html linebreaks
stop_words.add('br')
tokens_clean = [token for token in tokens_alpha if token not in stop_words]

print(f'Original tokens: {len(word_tokens)}')
print(f'After lowercasing: {len(tokens_lower)}')
print(f'After removing non-alpha: {len(tokens_alpha)}')
print(f'After removing stopwords: {len(tokens_clean)}')
print(f'\nCleaned tokens:\n{tokens_clean[:15]}')

## **2.0 Text Exploration - The Labeled Review Dataset** ##
## 2.1 Word Frequency Analysis  ##

In [ ]:
# Preprocess function for full corpus
def preprocess_text(text):

    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    return tokens

# Get all tokens from corpus
all_tokens = []

for text in df_labeledMovieReviews['review']:
    all_tokens.extend(preprocess_text(text))

# Count word frequencies
word_freq = Counter(all_tokens)
top_20 = word_freq.most_common(20)

# Display top words
freq_df = pd.DataFrame(top_20, columns=['word', 'count'])
freq_df

#### Word Frequency Analysis and Observations
There is a high frequency of the token "**br**"</br>
Upon further examination of the original data set this because html linebreaks are embedded in the reviews.
This has be cleaned up as it could/will skew any training/predictions we do.

### 2.2 Word Cloud Visualization

In [ ]:
# Create word cloud
wordcloud = WordCloud(
                        width=800,
                        height=400,
                        background_color='white',
                        colormap='Greys'
                    ).generate_from_frequencies(word_freq)

plt.figure(figsize=(10, 5))
plt.title('Word cloud of movie reviews corpus')
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

### 2.3 Word Frequency Distribution

In [ ]:
# Histogram of word frequency distribution
word_counts = list(word_freq.values())

plt.figure(figsize=(10, 5))
plt.title('Distribution of word frequencies')
plt.hist(word_counts, bins=100, color='grey', edgecolor='black')
plt.xlabel('Word frequency')
plt.ylabel('Number of words')
plt.yscale('log')  # Use logarithmic scale for better visibility
plt.tight_layout()
plt.show()

## **3.0 Sentiment Analysis** ##
Build sentiment classification models (positive:1 vs negative:0) using the following models:
- Naive Bayes
- Custom LSTM Deep Learning Model
- Pre-trained BERT model from HuggingFace

### 3.1 Naive Bayes Model
This was taken directly from George's Lesson_38_demo notebook.
#### Prepare the data

In [ ]:
# Prepare data
X = df_labeledMovieReviews['review']
y = df_labeledMovieReviews['sentiment']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
                                                        X, y,
                                                        test_size=0.2,
                                                        random_state=315
                                                    )

# Vectorize text using bag-of-words
vectorizer = CountVectorizer(max_features=5000, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f'Training set: {X_train_vec.shape}')
print(f'Test set: {X_test_vec.shape}')

#### Train the Model

In [ ]:
# Train Naive Bayes classifier
nb_classifier = MultinomialNB()
nb_classifier.fit(X_train_vec, y_train)

# Predict and evaluate
y_pred = nb_classifier.predict(X_test_vec)
accuracy = accuracy_score(y_test, y_pred)

print(f'Accuracy: {accuracy:.3f}\n')
print('Classification report:')
print(classification_report(y_test, y_pred))

### 3.2 VADER Sentiment Analyzer
In keep with the tme from Lesson_38_demo, I thought it would be interesting to explore Rules Based Sentiment Analysis thru VADER.</br>
Once again this was pretty much taken verbatim from George's Lesson_38_demo notebook.

****VADER** (Valence Aware Dictionary and sEntiment Reasoner) is a lexicon-based sentiment analyzer that uses a dictionary of words with pre-assigned sentiment scores.</br>
NLTK [VADER](https://www.nltk.org/howto/sentiment.html) documentation </br>

#### Sentiment Analysis for random sample from that the dataset.

In [ ]:
# Initialize VADER
sia = SentimentIntensityAnalyzer()

# Take 25 random samples from the train split and run thru VADER 
# just to see how they get ranked (satisfy curiosity)

# Analyze sentiment
print('Movie Review Random Sample VADER sentiment scores:\n')

for index, row in df_labeledMovieReviews.sample(25).iterrows():
    scores = sia.polarity_scores(row['review'])
    print(f'Text: {textwrap.fill(row["review"], width=120)}')
    print()
    print(f'Sentiment: {row["sentiment"]}')
    print(f'Sample #{index+1} Scores: {scores}\n')
    print()

#### Apply Vader to test set and compare with feature label

In [ ]:
# Apply VADER to test set
def vader_predict(text):

    scores = sia.polarity_scores(text)
    if scores['compound'] >= 0.05:
        return 1
    elif scores['compound'] <= -0.05:
        return 0
    else:
        return 0  # Default to negative for neutral

# Predict with VADER
y_pred_vader = [vader_predict(text) for text in X_train]
vader_accuracy = accuracy_score(y_train, y_pred_vader)

print(f'VADER accuracy: {vader_accuracy:.3f}')
print(f'Naive Bayes accuracy: {accuracy:.3f}')


#### VADER Analysis
Naive Bayes scored much higher than VADER in terms of Test dataset accuracy.</br>
This expected because **Naives Bayes** is supervised learning while **VADER** is rule based using a dictionary of predefined words and intensity scores. </br>
**Naive Bayes** is a trained model that actually learns, while **VADER** does not.
</br>

### 3.3 Custom Deep Learning model including LSTM layers

### 1.2 Initial Exploratory Data Analysis ###

In [ ]:
# print out the dataframes stats
print('GrammarAndReview Data')
PrintDataFrameStatistics(df_labeledMovieReviews)

#### Initial EDA Observations
##### **Missing Data** #####
- There are features with missing data in the dataset. For the purposed of our tasks, they won't affect us.</br>
    For purposes of this Capstone, we are interested in 3 features:
    - **reviews.text**
        - 35 missing rows - 0.05% of rows missing review.text
            - Course of action: Since it's such a small number, drop those rows from our dataset
    - **reviews.rating**
        - 0 missing rows - 0.00% of rows missing review.rating
            - Course of Action: No action required
    - **reviews.doRecommend**
        - 10615 missing rows - 14.94% of rows missing review.doRecommend
            - Course of Action: That is a significant enough number of missing values that we should do something about it.
                Set the missing values based on thresholds of review.rating

#### Additional Observations ####

##### Analylze Correct Punctuation Usage #####
- After a quick check, there is python library that does punctuation/grammatical analysis
- language_tool_python is a wrapper for LanguageTool

In [ ]:
# Check Punctuation Correctness

language_tool = language_tool_python.LanguageTool('en-US')

# I can't claim knowing these, credit where credit is due, I found these thru a quick search
df_punctuation = pd.DataFrame()

df_punctuation['missing_punctuation'] = ~df_grammarAndReviews['reviews.text'].str.strip().str.endswith(('.', '!', '?')).fillna(False)
df_punctuation['excessive_punctuation'] = df_grammarAndReviews['reviews.text'].str.contains(r'[!?]{3,}')
df_punctuation['missing_apostrophe'] = df_grammarAndReviews['reviews.text'].str.contains(r"\b(dont|cant|wont|isnt|didnt|doesnt)\b", case=False)

punct_cols = ['missing_punctuation', 'excessive_punctuation', 'missing_apostrophe']
df_punctuation[punct_cols].sum().plot(kind='bar', title='Punctuation Issues')
plt.ylabel('Count')
plt.show()

##### **Punctuation Examination Results** #####
- ~17% ( 12000/71044) of the ratings have missing punctuation (at end of review)

##### Spelling Analysis #####

In [ ]:
spellchecker_tool = SpellChecker()

def count_misspelled(text):
    if pd.isna(text):
        return 0
    words = text.split()
    return len(spellchecker_tool.unknown(words))

spelling_errors = df_grammarAndReviews['reviews.text'].apply(count_misspelled)

no_errors = (spelling_errors == 0).sum()

# Average
print('============================================================================')
print('Spelling Analysis Results')
print()
print(f'Total Number of Spelling Errors: {spelling_errors.sum()}')
print(f'Average Number of Spelling Errors per Review: {spelling_errors.mean()}')
print(f'Reviews with no Spelling Errors: {no_errors}')
print(f'Percentage of Reviews without Spelling Errors: {no_errors / len(spelling_errors) * 100:.2f}%')
print()
print('============================================================================')

df_spellingErrors = df_grammarAndReviews.copy(deep=True)
df_spellingErrors['spelling_errors'] = spelling_errors

df_spellingErrors.groupby('reviews.rating')['spelling_errors'].mean().plot(kind='bar')
plt.title('Average Spelling Errors by Rating')
plt.ylabel('Average Spelling Errors')
plt.xlabel('Rating')
plt.show()

##### **Spelling Analysis Results** #####
- Lower ratings have a slightly higher average number of spelling errors

#### Rating Review Analysis ####

In [ ]:
# Average character length of a review
average_review_length = df_grammarAndReviews['reviews.text'].str.len().mean()

# Calculate average number of words per review
average_review_word_count = df_grammarAndReviews['reviews.text'].str.split().str.len().mean()

print('============================================================================')
print('Review Length Analysis Results')
print()
print(f'Average Review Length: {average_review_length:.2f}')
print(f'Average Word Count per Review: {average_review_word_count:.2f}')
print()
print('============================================================================')


df_reviewLength = df_grammarAndReviews.copy(deep=True)
df_reviewLength['average_review_length'] = average_review_length
df_reviewLength['average_review_word_count'] = average_review_word_count

df_grammarAndReviews.groupby('reviews.rating')['reviews.text'].apply(
                                                                        lambda x: x.str.len().mean()
                                                                    ).plot(kind='bar')
plt.title('Average Review Length by Rating')
plt.ylabel('Average Review Length')
plt.xlabel('Rating')
plt.show()

df_grammarAndReviews.groupby('reviews.rating')['reviews.text'].apply(
                                                                        lambda x: x.str.split().str.len().mean()
                                                                    ).plot(kind='bar')
plt.title('Average Review Word Count by Rating')
plt.ylabel('Average Word Count')
plt.xlabel('Rating')
plt.show()

##### **Review Length Analysis** #####
- Negative reviews tend to be longer than positive reviews.

## 2. **Preprocess the Data** ##
#### Impute Missing Data ####

In [ ]:
# Drop rows with missing review.text feature
df_grammarAndReviews = df_grammarAndReviews[
                                            df_grammarAndReviews['reviews.text'].notna() & 
                                            (df_grammarAndReviews['reviews.text'].str.strip() != '')
                                        ]

# Impute missing doRecommend values based on reviews.rating - they're strongly correlated
# Arbitrarily use the following thresholds:
#   reviews.rating >= 3  ==> reviews.doRecommend = True
#   reviews.rating < 3  ==> reviews.doRecommend = False
#

df_grammarAndReviews.loc[
                        df_grammarAndReviews['reviews.doRecommend'].isna() & 
                        (df_grammarAndReviews['reviews.rating'] >= 3), 'reviews.doRecommend'] = True

df_grammarAndReviews.loc[
                        df_grammarAndReviews['reviews.doRecommend'].isna() & 
                        (df_grammarAndReviews['reviews.rating'] < 3), 'reviews.doRecommend'] = False

print('GrammarAndReview Data after imputed missing values completed')
#PrintDataFrameStatistics(df_grammarAndReviews)  

#### Label Encoding ####
- review.doRecommend is a categorical value (True/False) and should be mapped into numeric (integer) values
    This will be useful when we compare the end predictions against the intended/actual labels        

In [ ]:
df_grammarAndReviews['reviews.doRecommend'] = df_grammarAndReviews['reviews.doRecommend'].map(
                                                                                                {'True': 1, 'False': 0, True: 1, False: 0}
                                                                                            ).astype('int8')

#### Data Cleansing ####
1. Normalize the Text - translate from Unicode to Normalized Text
2. Remove Stop Words
3. Stem the Words
4. Lemmatize  words
5. Pad Reviews for consistent review lengths </br>

Once again I can't take credit for these methods.</br>
It easy to do a quick search/google to find the correct techniques

In [ ]:
# 1. Normalize the Text
def normalize_text(text):
    if pd.isna(text):
        return text
    normalized = unicodedata.normalize('NFKD', text)
    return normalized.encode('ascii', 'ignore').decode('ascii')

df_grammarAndReviews['reviews.text'] = df_grammarAndReviews['reviews.text'].apply(normalize_text)

# 2. Remove Top Words
def remove_stopwords(text):
    if pd.isna(text):
        return text
    words = text.split()
    return ' '.join([word for word in words if word.lower() not in stop_words])

df_grammarAndReviews['reviews.text'] = df_grammarAndReviews['reviews.text'].apply(remove_stopwords)

#3. Word Stemming
stemmer_tool = PorterStemmer()

def stem_text(text):
    if pd.isna(text):
        return text
    words = text.split()
    return ' '.join([stemmer_tool.stem(word) for word in words])

df_grammarAndReviews['reviews.text'] = df_grammarAndReviews['reviews.text'].apply(stem_text)

# 4. Lemmatize the words
nltk.download('wordnet')

lemmatizer_tool = WordNetLemmatizer()

def lemmatize_text(text):
    if pd.isna(text):
        return text
    words = text.split()
    return ' '.join([lemmatizer_tool.lemmatize(word) for word in words])

df_grammarAndReviews['reviews.text'] = df_grammarAndReviews['reviews.text'].apply(lemmatize_text)

# 5. Pad Reviews to have same lengths
max_length = df_grammarAndReviews['reviews.text'].str.len().max()
df_grammarAndReviews['reviews.text'] = df_grammarAndReviews['reviews.text'].str.pad(width=max_length, side='right', fillchar=' ')


#### **An Afterthought Regarding Padding** ####
- Padding the review text at this point is an unnecessary step.</br>
    What really matters is padding the review sequences generated by tokenization (occurs further in the processing)</br>
    Since I already did the work, I left it in for posterity's sake, and now I know how to do it.

##### **Count Plot Utility Functions** #####        

In [ ]:
def DisplayBarPlot(df_Target, x_axis, y_axis, 
                    x_axis_label=None,
                    y_axis_label='Count',
                    plot_title=None, 
                    colors=None):

    # Plot
    plt.figure(figsize=(12, 6))

    if colors:
        ax = sns.barplot(x=x_axis, 
                        y=y_axis, 
                        data=df_Target, 
                        hue=x_axis, 
                        palette=colors,
                        legend=False)
    else:
        ax = sns.barplot(x=x_axis, 
                        y=y_axis, 
                        data=df_Target, 
                        hue=x_axis, 
                        palette='Blues_d', 
                        legend=False)

    # Bold title and axis labels
    plt.title(plot_title, fontweight='bold')
    plt.xlabel(x_axis_label, fontweight='bold')
    plt.ylabel(y_axis_label, fontweight='bold')

    # Bold tick labels
    plt.xticks(rotation=45, ha='right', fontweight='bold')
    plt.yticks(fontweight='bold')

    # Add white count labels inside bars
    for container in ax.containers:
        ax.bar_label(container, color='white', fontweight='bold', label_type='center', fmt='%.0f')

    plt.tight_layout()
    plt.show()

def DisplayCountPlot(df_Target, x_axis, order, plot_title, fig_size, hue_value=None, plot_labels=None, displayLabels=True):

    if hue_value is not None:
        hue = hue_value
    else:
        hue = x_axis

    plt.figure(figsize=fig_size)
    ax = sns.countplot(x=x_axis, 
                       data=df_Target, 
                       order=order, 
                       hue=hue, 
                       palette='Blues_d', 
                       legend=False)

    plt.title(plot_title, fontweight='bold')
    plt.xlabel(x_axis, fontweight='bold')
    plt.ylabel('Count', fontweight='bold')
    plt.xticks(rotation=90, fontweight='bold')
    plt.yticks(fontweight='bold')

    # Skip bar labels for Year (too many bars)
    if displayLabels == True:
        for container in ax.containers:
            ax.bar_label(container, color='white', fontweight='bold', label_type='center')

    if plot_labels is not None:
        plt.legend(title=hue, labels=plot_labels)

    plt.tight_layout()
    plt.show()
    print()

#### Plot Distributions
- For purposes of this exercise, the only 2 distributions we want to plot are:
    - review.doRecommend
    - review.rating

In [ ]:
# Display Distributions

# review.ratings
ratingsOrder = df_grammarAndReviews['reviews.rating'].value_counts().index.sort_values()
DisplayCountPlot(df_grammarAndReviews, 'reviews.rating', ratingsOrder, 'Product Reviews', (14,6))

#review.doRecommend
recommendOrder = df_grammarAndReviews['reviews.doRecommend'].value_counts().index.sort_values()
DisplayCountPlot(df_grammarAndReviews, 'reviews.doRecommend', recommendOrder, 'Product Recommendations', (14,6))
 
# Correlation between rating and doRecommend
sns.countplot(x='reviews.rating', hue='reviews.doRecommend', data=df_grammarAndReviews, order=ratingsOrder)
plt.title('doRecommend by Rating')
plt.show()

##### **Observations from Distribution Plots**
- People who write reviews are more likely to write positive reviews - like the product, so more willing to take time to write a more thorough view
- People who submit feedback are more likely to write positive reviews/recommend the product
- As a result, the dataset is imbalanced as there are significantly more "**Recommend**" than "**Not Recommend**"
- To address this issue, we could assign higher weights to the minority class during training (greater penalty)
- If people have a negative experience with the product, they 'might be likely' to write a negative review/not recommend. (short and to the point)
    This is a slight bump in numbers, but nowhere near the positive experiences

#### Tokenize The Review Data ####
- I researched possible tokenizers and came up with two (2) options for tokenizing the review data
    (** Findings based on Google, Claude and ChatGPT)
    1. **Keras Tokenizer**
        - Simple word-level tokenization - more suited to RNN/LSTM models
    2. **HuggingFace Tokenizer(s)**    
        - Subword tokenization more suited for transformer models </br>

  Based on this information and ease of implementation, I decided to use the Keras tokenizer.

In [ ]:
# Tokenize the reviews

tokenizer_tool = Tokenizer()

tokenizer_tool.fit_on_texts(df_grammarAndReviews['reviews.text'])
sequences = tokenizer_tool.texts_to_sequences(df_grammarAndReviews['reviews.text'])
#print(f'sequences datatype: {type(sequences)}')

vocab_size = len(tokenizer_tool.word_index) + 1
print(f'vocab_size: {vocab_size}')

df_grammarAndReviews['tokens'] = sequences
df_grammarAndReviews['tokens'].head(20)

#### Test/Train Split #####
**Pad Sequenes** - before doing th Test/Train split, pad the sequences.</br>
We pad the sequences because Neural Networks require fixed sized input tensors. </br>
Padding sequences add zeros (0s) to fill out sequences so they are the same size.</br>
</br>
**Note**: Do I have to worry about introducing data leakage if I build vocab </br>
 and do padding before test/train split?

In [ ]:
# Extract X and y - pad sequences

# Do we REALLY need to have a max length here, or is this just for computation purposes?
max_len = 50

X = pad_sequences(sequences, maxlen=max_len, padding='post', value=0)
y = df_grammarAndReviews['reviews.doRecommend'].values

# Then split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'Score distribution (train): {Counter(y_train)}')

## **3.0 Model Creation and Training** ##
#### Build The Model ####
- We are solving a **Binary Classification** Problem:
    - Do Recommend == 1
    - Do Not Recommend == 0 </br>

For **Binary Classification** we use:
- **Sigmoid** as the Activation Function on the Dense (last) layer of the model (b/w 0 and 1)
- **binary_crossentropy** as the loss function on the model

##### **Training and Evaluation Utility Functions** ####

In [ ]:
########################################
##
## RNN Model Template from class demo
##
#######################################
# Model parameters
# hidden_dim = 8 # RNN hidden state size

# model = Sequential([
#     SimpleRNN(hidden_dim, input_shape=(seq_length, vocab_size)),
#     Dense(vocab_size, activation='softmax')
# ])



###############################################
# We're solving Binary Classification problem
###############################################

embedding_dim = 128

def CreateRNNModel(dropout_rate):

    model = Sequential([

        # Embedding - convert 2D sequences into 3D vectors [required by SimpleRNN(), LTSM()]
        Embedding(vocab_size, embedding_dim, input_length=max_len),
        
        # Block 1: Convolutional
        # Text is 1 dimensional sequence - so use 1D function layers
        # relu activation - 
        Conv1D(filters=64, kernel_size=3, activation='relu'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(dropout_rate),
        
        # Block 2: Recurrent
        SimpleRNN(64, return_sequences=False),
        Dropout(dropout_rate),
        
        # Block 3: Dense
        Dense(32, activation='relu'),
        Dropout(dropout_rate),

        # Binary classification: doRecommend 0 or 1
        Dense(1, activation='sigmoid')  
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    model.summary()
    return model

def CreateLTSMModel(dropout_rate):
    model = Sequential([

        # Embedding - convert 2D sequences into 3D vectors [required by SimpleRNN(), LTSM()]
        Embedding(vocab_size, embedding_dim, input_length=max_len),
        
        # Block 1: Convolutional
        # Text is 1 dimensional sequence - so use 1D function layers
        # relu activation - 
        Conv1D(filters=64, kernel_size=3, activation='relu'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(dropout_rate),
        
        # Block 2: Recurrent
        LSTM(64, return_sequences=False),
        Dropout(dropout_rate),
        
        # Block 3: Dense
        Dense(32, activation='relu'),
        Dropout(dropout_rate),

        # Binary classification: doRecommend 0 or 1
        Dense(1, activation='sigmoid')  
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    model.summary()
    return model


#### Train the Model ####

In [ ]:
# Train the model
def TrainTheModel(model, dropout_rate, epochs):

    sequential_history = model.fit(
                                    X_train, 
                                    y_train,
                                    epochs=epochs,  # 10
                                    batch_size=32,
                                    validation_split=0.2
                                )

    print(f'dropout_rate: {dropout_rate}  loss: {sequential_history.history["loss"]}')
    print(f'dropout_rate: {dropout_rate}  accuracy: {sequential_history.history["accuracy"]}')
    print(f'dropout_rate: {dropout_rate}  val_loss: {sequential_history.history["val_loss"]}')
    print(f'dropout_rate: {dropout_rate}  val_accuracy: {sequential_history.history["val_accuracy"]}')
    return sequential_history

#### Plot the Learning Curves ####

In [ ]:
def PlotLearningCurves(history, plot_title):

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))

    axes[0].set_title(f'{plot_title}: loss')
    axes[0].plot(history['loss'], label='Training')
    axes[0].plot(history['val_loss'], label='Validation')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss (binary crossentropy)')
    axes[0].legend(loc='best')

    axes[1].set_title(f'{plot_title}: accuracy')
    axes[1].plot(history['accuracy'], label='Training')
    axes[1].plot(history['val_accuracy'], label='Validation')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend(loc='best')

    plt.tight_layout()
    plt.show()

##### **CNN-LSTM - Loss and Accuracy Analysis** ####

#### Test Set Evaluation ####

In [ ]:
def EvaluateModel(model, dropout_rate):
    # Evaluate against test set
    test_loss, test_accuracy = model.evaluate(X_test, y_test)
    print(f'dropout_rate: {dropout_rate}  Test Loss: {test_loss:.4f}')
    print(f'dropout_rate: {dropout_rate}  Test Accuracy: {test_accuracy:.4f}')


#### Confusion Matrix Plots ####

In [ ]:
def PlotConfusionMatrix(model, X_test, y_test, plot_title):

    y_probs = model.predict(X_test).flatten()
    y_pred = (y_probs >= 0.5).astype(int)

    print('----------------------------------------------------------')
    print(plot_title)
    print()
    print(classification_report(y_test, y_pred, target_names=['Not Recommend', 'Recommend']))
    print(f'ROC-AUC: {roc_auc_score(y_test, y_probs):.4f}')
    print('----------------------------------------------------------')

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Recommend', 'Recommend'])
    disp.plot(cmap='Blues')
    plt.title(plot_title)
    plt.show()

#### Train/Test Wrapper Methods ####

In [ ]:
# For collecting all results
results = []

def ProcessRNNModel(dropout_rate, epochs):

    model = CreateRNNModel(dropout_rate)
    history = TrainTheModel(model, dropout_rate, epochs)
    PlotLearningCurves(history.history, f'CNN-RNN: dropout_rate={dropout_rate}')
    EvaluateModel(model, dropout_rate)
    PlotConfusionMatrix(model, X_test, y_test, plot_title=f'CNN-RNN Confusion Matrix: dropout_rate: dropout_rate: {dropout_rate}')

    y_probs = model.predict(X_test).flatten()
    y_pred = (y_probs >= 0.5).astype(int)
    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

    results.append({
                    'model_name': 'CNN-RNN',
                    'dropout_rate': dropout_rate,
                    'epochs': epochs,
                    'test_loss': test_loss,
                    'test_accuracy': test_accuracy,
                    'precision': precision_score(y_test, y_pred),
                    'recall': recall_score(y_test, y_pred),
                    'f1': f1_score(y_test, y_pred),
                    'roc_auc': roc_auc_score(y_test, y_probs)
                })
    
    print()

def ProcessLTSMModel(dropout_rate, epochs):
    
    model = CreateLTSMModel(dropout_rate)
    history = TrainTheModel(model, dropout_rate, epochs)
    PlotLearningCurves(history.history, f'CNN-LTSM: dropout_rate="{dropout_rate}')
    EvaluateModel(model, dropout_rate)
    PlotConfusionMatrix(model, X_test, y_test, plot_title=f'CNN-LTSM Confusion Matrix: dropout_rate: dropout_rate: {dropout_rate}')

    y_probs = model.predict(X_test).flatten()
    y_pred = (y_probs >= 0.5).astype(int)
    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

    results.append({
                    'model_name': 'CNN-LSTM',
                    'dropout_rate': dropout_rate,
                    'epochs': epochs,
                    'test_loss': test_loss,
                    'test_accuracy': test_accuracy,
                    'precision': precision_score(y_test, y_pred),
                    'recall': recall_score(y_test, y_pred),
                    'f1': f1_score(y_test, y_pred),
                    'roc_auc': roc_auc_score(y_test, y_probs)
                })

    print() 

#### Run The Models for RNN and LTSM for epoch/dropout_rate Combinations ####

In [ ]:
epochs = 10
dropout_rates = [0.2, 0.3, 0.4, 0.5]

print()
print('@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@')
print('RNN Models Processing')
print()
print('@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@')
print()
# RNN
for dropout_rate in dropout_rates:
    print(f'\n=== Dropout Rate: {dropout_rate} ===')
    ProcessRNNModel(dropout_rate, epochs)

print()
print('@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@')
print('LSTM Models Processing')
print()
print('@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@')
print()

# LSTM
for dropout_rate in dropout_rates:
    print(f'\n=== Dropout Rate: {dropout_rate} ===')
    ProcessLTSMModel(dropout_rate, epochs)

# Convert all results to a dataframe
df_results = pd.DataFrame(results)    

print()
print('@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@')
print('Model Processing Complete')
print()
print('@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@')
print()

#### RNN-CNN and LTSM Analysis ####
##### Loss, Accuracy and ConfusionMatrix Analysis Across Dropout Rates #####
- **RNN-CNN**: 
    - Dropout Rate: 0.2
        - Learning Curves: training/validation track closely, no signficant gap. Accuracy peaks ~91%. Not overfitting.
        - ConfusionMatrix: Does well on correct Recommended, but fails on Not Recommended
    - Dropout Rate: 0.3
        - Learning Curves: Loss increases after 8 epochs, while accuracy increases. Overfitting.
        - ConfusionMatrix: Same issues
    - Dropout Rate: 0.4
        - Learning Curves: Spikes in validation. Huge decrease and upward bounce in Validation Accuracy at 8 epochs. (too large of a dropout_rate??)
        - ConfusionMatrix: Same issues
    - Dropout Rate: 0.5
        - Learning Curves: Training/Validation track close with no significant gaps. Loss spike at 7 epochs. Slow learning (large dropout_rate?). Not overfitting
        - ConfusionMatrix: Same issues   
- **LTSM-CNN**: 
    - Dropout Rate: 0.2
        - Learning Curves: Overfitting. Training loss decrease while validation loss increases. Training accuracy high while validation low and fluctuates (large gap).
        - ConfusionMatrix: Performs better than RNN-CNN. More false Recommendeds and more postive Not Recommendeds
    - Dropout Rate: 0.3
        - Learning Curve: Similar to 0.2
        - ConfusionMatrix: Performs slightly worse for false Not Recommendeds
    - Dropout Rate: 0.4
        - Learning Curves: Similar to 0.2
        - ConfusionMatrix: Performs better for positive Not Recommendeds
    - Dropout Rate: 0.5
        - Learning Curves: Severe overfitting - large gaps. Validation looks unstable/jagged
        - ConfusionMatrix: Performs worse than 0.4                        
</br>
##### Display Top Sorted Metrics and Plot Metric Comparisons Between **RNN-CNN** and **LSTM-CNN** #####


In [ ]:
print('=================================================================')
print('Best Model Scores')
print('=================================================================')
print()

print('=== By Accuracy ===')
print(df_results.sort_values('test_accuracy', ascending=False).to_string())

print('\n=== By Loss (lower is better) ===')
print(df_results.sort_values('test_loss', ascending=True).to_string())

print('\n=== By Precision ===')
print(df_results.sort_values('precision', ascending=False).to_string())

print('\n=== By Recall ===')
print(df_results.sort_values('recall', ascending=False).to_string())

print('\n=== By F1 ===')
print(df_results.sort_values('f1', ascending=False).to_string())

print('\n=== By ROC-AUC ===')
print(df_results.sort_values('roc_auc', ascending=False).to_string())


# Plot the results
metrics = ['test_accuracy', 'test_loss', 'precision', 'recall', 'f1', 'roc_auc']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, metric in enumerate(metrics):
    df_pivot = df_results.pivot(index='dropout_rate', columns='model_name', values=metric)
    df_pivot.plot(kind='bar', ax=axes[i])
    axes[i].set_title(metric)
    axes[i].set_xlabel('Dropout Rate')
    axes[i].set_ylabel(metric)
    axes[i].legend(title='Model')
    axes[i].tick_params(axis='x', rotation=0)

plt.suptitle('CNN-RNN vs CNN-LSTM: Metric Comparisons Across Dropout Rates', fontsize=14)
plt.tight_layout()
plt.show()

#### Metrics Analysis ####
- The dataset is definitely unbalanced - as evidenced by looking at ConfusionMatrices - "**Do Recommends**"</br>
With more time I could have played with class weighting to see if that helped things out.
- **F1 Score** is the metric of choice because this is a **Classification Problem** with **imbalanced classes**.

#### **Best Overall Performing Model/Dropout Rate Combination** ####
| model_name | dropout_rate | epochs | test_loss | test_accuracy | precision | recall | f1 | roc_auc |
|------------|--------------|--------|-----------|--------------|----------|--------|------|---------|
| CNN-LSTM   | 0.2          | 10     | 0.298357  | 0.931841     | 0.960210 | 0.964972 | 0.962585 | 0.885313 |

</br>

## **CNN-LSTM** with **dropout_rate=0.2**
Is the best model/dropout rate combination. 
- Highest F1 Score
- Reasonably high ROC-AUC
- Lowest loss
- Reasonably high accuracy without overfitting

##### **Final Thoughts and Observations** #####
First thing I learned with this assignment is that there is a lot of information into the target participants that you can glean with some </br>
simple straightforward analysis of the text itself (spelling, punctionation, length, et al). There is also a fair amount of preprocessing</br>
required to get the text into something that's normalized and easily consumed by models (tokenizing - models like working with numbers)v
With more time, it would have been fun to look at GRU and play with hyperparameter tuning to get better performance. </br>
We could have even factored some of the other features into the model, but hat was not the main goal of this exercise. </br>
The main goal was to see how we could combine RNN and CNN to build a predictive model based on text fields.</br>
</br>
.. **and finally**, I wish I had my new GPU based machine (either *Mr. Windows Desktop* or *Mr. Mc-Linux-Beast*) to make the processing of the assignments more efficient.
